# Create a SageMaker Pipeline to Automate All the Steps from Data Prep to Model Deployment

Reference: 
- https://sagemaker-examples.readthedocs.io/en/latest/end_to_end/fraud_detection/pipeline-e2e.html
- https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html


Steps:
- Claims Data Wrangler Preprocessing Step
- Customers Data Wrangler Preprocessing Step
- Create Dataset and Train/Test Split
- Train XGBoost Model
- Model Pre-Deployment Step
- Run Bias Metrics with Clarify
- Register Model
- Deploy Model
- Combine and Run the Pipeline Steps

**Good Practice Habits:**
- Keep a unique name for the pipeline i.e."pipeline-name-timestamp" and enforce this name for everything i.e. process names, model names, ouptuts etc. So that tracebility achieved.
- When you give a s3 output dir uri as output path or ProcessingOutput or other outputs, always provide "pipeline-name-timestamp" prefix at the end so all things will go inside that directory.
- Still I see, ProcessJob or TrainingJob always create a different s3 dir to keep the code.

In [1]:
import sagemaker 
import boto3 
import pandas as pd 
from datetime import datetime

/opt/conda/lib/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
REGION = sagemaker.session.Session().boto_region_name
print("REGION: ", REGION) 

boto3_session = boto3.Session(region_name=REGION)

sagemaker_boto3_client = boto3_session.client("sagemaker")
s3_boto3_client = boto3_session.client("s3")
sagemaker_session = sagemaker.session.Session(boto_session=boto3_session, sagemaker_client=sagemaker_boto3_client)

BUCKET = sagemaker_session.default_bucket()
PREFIX = "FraudDetection_AutoInsurance"

ROLE=sagemaker.get_execution_role()
print("ROLE: ", ROLE)
print("BUCKET: ", BUCKET) 
print("PREFIX: ", PREFIX) 

s3_dir_uri = f"s3://{BUCKET}/{PREFIX}"
print(s3_dir_uri)

REGION:  us-east-1
ROLE:  arn:aws:iam::205930620783:role/service-role/AmazonSageMaker-ExecutionRole-20250401T145997
BUCKET:  sagemaker-us-east-1-205930620783
PREFIX:  FraudDetection_AutoInsurance
s3://sagemaker-us-east-1-205930620783/FraudDetection_AutoInsurance


## Pipeline Parameters

Pipeline parameters are conceptually similar to command-line arguments (argparse) in a Python script. Both allow external users or systems to provide input values at runtime instead of hardcoding them.

As well, unlike command line args, these "Parameters" are automatically logged and tracked. 


In [3]:
from sagemaker.workflow.parameters import (    
    ParameterInteger, ParameterFloat, ParameterString
)

# current timestamp
current_timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S") 
print(current_timestamp)

# Pipeline Name and it's execution name
pipeline_name = "FraudDetection-AutoInsurance"
pipeline_execution_name = f'{pipeline_name}-{current_timestamp}'

# Data Processing Parameters
p_s3_data_inp_uri = ParameterString(name="s3DataDirURI", default_value=f'{s3_dir_uri}/data/processed_data/dataset.csv')
s3_dataprocess_out_dir_uri = f'{s3_dir_uri}/01_dataprocessing_jobs/{pipeline_execution_name}'
print(s3_dataprocess_out_dir_uri)

processing_instance_type = "ml.m5.xlarge"# "ml.t2.medium" is not supported
train_data_uri = f"{s3_dir_uri}/data/train.csv"
test_data_uri = f"{s3_dir_uri}/data/test.csv"
print(train_data_uri)

# Training Parameters
s3_estimator_out_dir_uri = f'{s3_dir_uri}/02_training_jobs/{pipeline_execution_name}'#ParameterString(name="s3EstimatorDirURI", default_value=f'{s3_dir_uri}/training_jobs')
print(f'{s3_dir_uri}/training_jobs')

train_instance_type = "ml.c5.xlarge"
train_instance_count = 1
target_var = "fraud"

# Batch Transformation Test Data
batch_transform_instance_count=1
batch_transform_instance_type='ml.m5.large' 
s3_batch_transform_output_uri = f'{s3_dir_uri}/04_transform_jobs/{pipeline_execution_name}'

# Model Registry Parameters
p_model_appoval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")


2025-06-19-12-30-27
s3://sagemaker-us-east-1-205930620783/FraudDetection_AutoInsurance/01_dataprocessing_jobs/FraudDetection-AutoInsurance-2025-06-19-12-30-27
s3://sagemaker-us-east-1-205930620783/FraudDetection_AutoInsurance/data/train.csv
s3://sagemaker-us-east-1-205930620783/FraudDetection_AutoInsurance/training_jobs


## Data Preprocessing Step

For now create a step which returns the path of the processed train and test data csv on s3.

### sagemaker.processing.Processor v/s  sagemaker.sklearn.processing.SKLearnProcessor
-  **Processor** is base class in the Sagemaker SDK used to run arbritary processing jobs. It gives you full control, i.e. which docker image to choose, entry point, env vars. You must configure everything including the image uri.
-  **SKLearnProcessor** is a pre-configured sub class of Processor. Automaticallt sets the scikit-learn image, handles dependencies. Designed specifically for scikit learn based workflows.

**Notes:**
- There is not pre-defined directory structure or CHANNELS as Estimator class for Processor class. It will create a dir `input`, `output` or `custom_name` under `/opt/ml/processing/` if mentioned in the ProcessingInput or ProcesingOutput.
- You need to create the sub-directories manually if you are trying to access from inside the script and not mentioned in ProcessingInput or Output.
- There can be multiple files written in the same dir i.e. test.csv and test_to_predict.csv in `/opt/ml/processing/output/test` and it will write all files in the s3 uri path mantaining the sub-directory structure.
- To reference (down the pipeline) the specific s3 dir where outputs are written use `dataset_step_01.properties.ProcessingOutputConfig.Outputs["test_to_predict"].S3Output.S3Uri`
- But to reference the specific s3 file, use `Join(on="/", values=[dataset_step_01.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri, "train.csv"])` where `Join` is `sagemaker.workflow.functions.Join`.
- SKLearnProcessor does not take 'output_path', so the outputs will go in default s3 bucket under "data_process_name-timestamp_or_randomchars". You can though direct the `/opt/ml/processing/output/` to a s3 bucket using `ProcessingOutput`. But still, ProcessorJob writes the script code in the default bucket and am not sure how to re-reoute it to differner place.

#### You can not specify individual paths of filenames in the destination of ProcessingOutput or ProcessingInput, only specify S3 dir. Why?
SageMaker ProcessingOutput is designed to upload the entire contents of a directory from the container (source) to a destination S3 prefix (destination). It doesn't support targeting specific file names because:
- The actual file names are unknown until your script runs.
- Multiple files are often written (e.g., train/test splits, metadata, etc.).
- It's intended to preserve folder structure and be reusable downstream


In [4]:
from sagemaker.sklearn.processing import SKLearnProcessor 
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

# first configure the SKLearnProcessor Class
data_processor = SKLearnProcessor(
    framework_version='0.23-1',
    role=ROLE,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name=pipeline_execution_name,
    sagemaker_session=sagemaker_session
)

dataset_step_01 = ProcessingStep(
    name="ProcessData",
    processor=data_processor,
    code="scripts/data_processing_script.py",
    inputs=[
        sagemaker.processing.ProcessingInput(source=p_s3_data_inp_uri, destination="/opt/ml/processing/input/") #mapping of source to destination: read operation that reads from the destination will be re-route to source.
    ],
    outputs=[
        # The SKLearnProcessor instance does not have the following dir structure like there is a predifined dir structure and CHANNELS in the Estimator class, atleast no 'test' dir. But when you mention it in ProcessingInput and ProcessingOutput it will create it. 
        # If there are other subdirectories that you access in the script but not mentioned in ProcessingInput/Output then you need to create it.

        # Uploads the content of output directory with structure retained on the s3 uri dir.
        sagemaker.processing.ProcessingOutput(output_name="output", source="/opt/ml/processing/output/", destination=f'{s3_dataprocess_out_dir_uri}/output'),
        # If you want to store the inputs for tracebility, But it raises an error as the input is already bound to the above s3 uri.
        #sagemaker.processing.ProcessingOutput(output_name="input", source="/opt/ml/processing/input/", destination=f'{s3_dataprocess_out_dir_uri}/input') # This doesn't work
    ]
    #job_arguments=[]
)

[06/19/25 12:30:27] INFO     Defaulting to only available Python version: py3                     ]8;id=383082;file:///opt/conda/lib/python3.12/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=295129;file:///opt/conda/lib/python3.12/site-packages/sagemaker/image_uris.py#610\610]8;;\

## Training Step


In [5]:
dataset_step_01.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri

{'_step': <sagemaker.workflow.steps.ProcessingStep object at 0x7fe6b2ff42c0>, 'step_name': 'ProcessData', 'path': "ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri", '_shape_names': ['S3Uri'], '__str__': 'S3Uri'}

In [6]:
from sagemaker.xgboost.estimator import XGBoost 
from sagemaker.workflow.functions import Join

xgb_estimator = XGBoost(
    framework_version="1.0-1",
    entry_point="scripts/xgboost_model_script.py",
    output_path=s3_estimator_out_dir_uri,       # For model file and metrics file. 
    code_location=s3_estimator_out_dir_uri, # If not provided it will be put in the default bucket
    hyperparameters={'target-var':target_var, 'max-depth':6, 'eta':0.3, 'objective':'binary:logistic', 'num-boost-round':100, 'nfold':5},
    role=ROLE,
    instance_count=train_instance_count,
    instance_type=train_instance_type,
    base_job_name=pipeline_execution_name
)

#xgb_estimator.fit(inputs={'train':train_data_uri, 'test':test_data_uri})
train_step_02 = TrainingStep( 
    name="TrainingJob",
    estimator=xgb_estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(s3_data=Join(on="/", values=[dataset_step_01.properties.ProcessingOutputConfig.Outputs["output"].S3Output.S3Uri, "train.csv"]))
    },
    # There is no TrainingOutput like ProcessingOutput class for Training Step
    depends_on=[dataset_step_01.name]
)

## Model Pre-Deployment Step. Create a Model from the Training Job

This step creates a model object that can be reused in multiple downstream steps like deploy, register, batch transformation etc.

**Notes:** 
- We are only declaring the instance type this step never deploys the model on the instance.
- Do not use XGBoostModel as it doesn't understand dynamic pipeline properties.

In [7]:
from sagemaker.model import Model # You can also use sagemaker.transformer.Transformer directly instead of creating a model and then calling the transformer
from sagemaker.workflow.steps import CreateModelStep, TransformStep

# Do not use XGBoostModel as it doesn't understand the pipeline output s2 uri. It only understand string s3 uri.
xgboost_model = Model(
    name=pipeline_execution_name,
    role=ROLE,
    sagemaker_session=sagemaker_session,
    image_uri=train_step_02.properties.AlgorithmSpecification.TrainingImage,
    model_data=train_step_02.properties.ModelArtifacts.S3ModelArtifacts
    #entry_point
)

# create a model from the training 
model_creation_step_03 = CreateModelStep(
    name="CreateModel",
    model=xgboost_model,
    inputs=sagemaker.inputs.CreateModelInput(instance_type="ml.m5.large") # We are not deploying the model yet, but we still need to declare the instance type for future deployments.   
)


## Batch Transform for Batch Prediction on the test data
It's a batch transform job, that runs on an instance. 

**Note**: 
- Do not use xgboost_model.transformer, as it doesnot understand dynamic pipeline variables.
- Remove the target factor and column header from the csv input file and data. That's how the transformer takes the data


In [8]:
dataset_step_01.properties.ProcessingOutputConfig.Outputs["test_to_predict"].S3Output.S3Uri

{'_step': <sagemaker.workflow.steps.ProcessingStep object at 0x7fe6b2ff42c0>, 'step_name': 'ProcessData', 'path': "ProcessingOutputConfig.Outputs['test_to_predict'].S3Output.S3Uri", '_shape_names': ['S3Uri'], '__str__': 'S3Uri'}

In [9]:
from sagemaker.transformer import Transformer


xgb_transformer = Transformer(
    model_name=model_creation_step_03.properties.ModelName,
    instance_count=batch_transform_instance_count,
    instance_type=batch_transform_instance_type,
    output_path=s3_batch_transform_output_uri,   # Provide output data path for predictions or it will ouptut in default bucket
    strategy="SingleRecord",                     # How to predict multiple record or single
    assemble_with="Line",                         # How to join multiple requests
    accept="text/csv",
    base_transform_job_name=pipeline_execution_name
)

batch_transformer_step_04 = TransformStep(
    name="BatchTransformStep",
    transformer=xgb_transformer,
    inputs=sagemaker.inputs.TransformInput(
        data=Join(on="/", values=[dataset_step_01.properties.ProcessingOutputConfig.Outputs["output"].S3Output.S3Uri, "test_to_predict.csv"]), 
        content_type="text/csv", 
        split_type="Line"
    )
)

## Evaluate the model
Use Sklearn processor for this, pass the actual target and the test predictions and calculate the AUC.

In [10]:
batch_transformer_step_04.properties.TransformOutput.S3OutputPath

{'_step': <sagemaker.workflow.steps.TransformStep object at 0x7fe6aeadfc20>, 'step_name': 'BatchTransformStep', 'path': 'TransformOutput.S3OutputPath', '_shape_names': ['S3Uri'], '__str__': 'S3Uri'}

In [11]:
from sagemaker.sklearn.processing import SKLearnProcessor 
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.properties import PropertyFile 


# first configure the SKLearnProcessor Class
model_evaluator = SKLearnProcessor(
    framework_version='0.23-1',
    role=ROLE,
    instance_type="ml.t3.medium",
    instance_count=1,
    base_job_name=pipeline_execution_name, # Anyhow it will be override by the ProcessingStep
    sagemaker_session=sagemaker_session
)

evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="eval", path="test_metrics.json"
)
# name (str): The name of the property file for reference with `JsonGet` functions.
# output_name (str): The name of the processing job output channel.
# path (str): The path to the file at the output channel location.

evaluatemodel_step_05 = ProcessingStep(
    name="EvaluateModel",
    processor=model_evaluator,
    code="scripts/model_evaluation_script.py",
    inputs=[
        # The SKLearnProcessor instance does not have the following dir structure like there is a predifined dir structure and CHANNELS in the Estimator class, atleast no 'test' dir. But when you mention it in ProcessingInput and ProcessingOutput it will create it. 
        # If there are other subdirectories that you access in the script but not mentioned in ProcessingInput/Output then you need to create it.
        sagemaker.processing.ProcessingInput(
            source=Join(on="/", values=[dataset_step_01.properties.ProcessingOutputConfig.Outputs["output"].S3Output.S3Uri, "test.csv"]),
            destination="/opt/ml/processing/test"
        ), #mapping of source to destination: read operation that reads from the destination will be re-route to source.
        sagemaker.processing.ProcessingInput(
            source=Join(on="/", values=[batch_transformer_step_04.properties.TransformOutput.S3OutputPath,"test_to_predict.csv.out"]),
            destination="/opt/ml/processing/test_predictions"
        ), #mapping of source to destination: read operation that reads from the destination will be re-route to source.
    ],
    outputs=[
        # Wtrite the evaluation metrics into the training job output
        sagemaker.processing.ProcessingOutput(output_name="eval", source="/opt/ml/processing/evaluation", destination=batch_transformer_step_04.properties.TransformOutput.S3OutputPath),
      ],
    property_files=[evaluation_report]
)


                    INFO     Defaulting to only available Python version: py3                     ]8;id=134681;file:///opt/conda/lib/python3.12/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=401344;file:///opt/conda/lib/python3.12/site-packages/sagemaker/image_uris.py#610\610]8;;\

In [28]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.step_collections import RegisterModel #Step

# If a package group is note created then first create that
# You need to update model metrics to0
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
    s3_uri=Join(on="/", values=[evaluatemodel_step_05.properties.ProcessingOutputConfig.Outputs["eval"].S3Output.S3Uri, 'metrics_model.json']),
    content_type="application/json"
    )
    # add bias parameter with sagemaker clarify outputs. And many more you can add
)

## Register The Model
register_model_step_06 = RegisterModel(
    name="RegisterTrainModelStep",
    model=model_creation_step_03.model,
    estimator=None, # If "model" is not given then give xgb_estimator from TrainStep
    content_types=["text/csv"],
    response_types=["text/csv"],
    approval_status=p_model_appoval_status,
    model_package_group_name="FraudDetection-AutoInsurance",
    model_metrics=model_metrics,
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"]
)


In [32]:
help(ProcessingStep)

Help on class ProcessingStep in module sagemaker.workflow.steps:

class ProcessingStep(ConfigurableRetryStep)
 |  ProcessingStep(name: str, step_args: Optional[sagemaker.workflow.pipeline_context._JobStepArguments] = None, processor: Optional[sagemaker.processing.Processor] = None, display_name: Optional[str] = None, description: Optional[str] = None, inputs: Optional[List[sagemaker.processing.ProcessingInput]] = None, outputs: Optional[List[sagemaker.processing.ProcessingOutput]] = None, job_arguments: Optional[List[str]] = None, code: Optional[str] = None, property_files: Optional[List[sagemaker.workflow.properties.PropertyFile]] = None, cache_config: Optional[sagemaker.workflow.steps.CacheConfig] = None, depends_on: Optional[List[Union[str, sagemaker.workflow.steps.Step, ForwardRef('StepCollection')]]] = None, retry_policies: Optional[List[sagemaker.workflow.retry.RetryPolicy]] = None, kms_key: Optional[str] = None)
 |
 |  `ProcessingStep` for SageMaker Pipelines Workflows.
 |
 |  M

## Combine and Run the pipeline steps.


In [29]:
# Composing and creating the pipeline
from sagemaker.workflow.pipeline import Pipeline


pipeline= Pipeline(
    name=pipeline_name,
    parameters=[p_s3_data_inp_uri, p_model_appoval_status],
    steps=[
        dataset_step_01,
        train_step_02,
        model_creation_step_03,
        batch_transformer_step_04,
        evaluatemodel_step_05,
        register_model_step_06
    ]
)

pipeline.upsert(role_arn=ROLE)
#import json
#print(json.loads(pipeline.describe()["PipelineDefinition"]))

[06/19/25 12:52:48] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=786013;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=972481;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=765109;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=904908;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelName' from the pipeline definition by default since ]8;id=180629;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=131171;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             it will be overridden at pipeline execution time. Please utilize the                  
                             PipelineDefinitionConfig to persist this field in the pipeline                        
                             definition if desired.                                                                

                    WARNING  Popping out 'TransformJobName' from the pipeline definition by        ]8;id=245085;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=417949;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=824143;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=776211;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

...

                    WARNING  Popping out 'TransformJobName' from the pipeline definition by        ]8;id=484698;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=16792;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=595192;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=858254;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition since   ]8;id=539449;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/_utils.py\_utils.py]8;;\:]8;id=499394;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/_utils.py#515\515]8;;\
                             it will be overridden in pipeline execution time.                                     

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=577643;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py\utilities.py]8;;\:]8;id=555270;file:///opt/conda/lib/python3.12/site-packages/sagemaker/workflow/utilities.py#465\465]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:205930620783:pipeline/FraudDetection-AutoInsurance',
 'ResponseMetadata': {'RequestId': 'f38cf333-f748-4736-817e-5c57854bb02c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'f38cf333-f748-4736-817e-5c57854bb02c',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '96',
   'date': 'Thu, 19 Jun 2025 12:52:49 GMT'},
  'RetryAttempts': 0}}

In [31]:
# run the pipeline
#parameters = {"s3DataDirURI": s3_data_dir_uri}
pipeline.start(execution_display_name=pipeline_execution_name)#parameters=parameters) # You can set parameters are the run time and it will override the default ones.

_PipelineExecution(arn='arn:aws:sagemaker:us-east-1:205930620783:pipeline/FraudDetection-AutoInsurance/execution/k8zpsq1dapq5', sagemaker_session=<sagemaker.session.Session object at 0x7fe6ae49fc20>)